# Supervised Learning
## 2.1 Classical ML Models (Logistic Regression & Gradient Boosted Tree)
We decided to mostly use the last-measured value as input feature as this the closest to death and should be most informative. The 4 static values will be kept (for static weight this is the first).

In [2]:
# Simple Feature Preprocessing
# For each patient: last-measured value of each dynamic variable + static variables

import numpy as np
import pandas as pd
import pickle

df_train = pd.read_parquet("processed/set_a_processed.parquet")
df_val   = pd.read_parquet("processed/set_b_processed.parquet")
df_test  = pd.read_parquet("processed/set_c_processed.parquet")

DYNAMIC_VARS = sorted([
    'ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol',
    'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT',
    'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP',
    'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2',
    'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight', 'pH'
])
STATIC_VARS = ['Age', 'Gender', 'Height', 'StaticWeight']
print(len(DYNAMIC_VARS))
print(len(STATIC_VARS))
ALL_FEATURES = DYNAMIC_VARS + STATIC_VARS  # 41 features total

def extract_simple_features(df): #extract the last observation of each patient("indicated by recordID"

    last_row = df.sort_values(["RecordID", "hour"]).groupby("RecordID").last().reset_index()
    
    features = last_row[["RecordID"] + ALL_FEATURES].copy()
    labels   = last_row[["RecordID", "In-hospital_death"]].copy()
    
    return features, labels

X_train, y_train = extract_simple_features(df_train)
X_val,   y_val   = extract_simple_features(df_val)
X_test,  y_test  = extract_simple_features(df_test)

print(y_train['In-hospital_death']) #check the prevalence rate to see check if we should include something to balance the data and what are good values for the

X_train.head()

37
4
0       0
1       0
2       0
3       0
4       0
       ..
3995    0
3996    0
3997    0
3998    1
3999    0
Name: In-hospital_death, Length: 4000, dtype: int64


,RecordID,ALP,ALT,AST,Albumin,BUN,Bilirubin,Cholesterol,Creatinine,DiasABP,...,TroponinI,TroponinT,Urine,WBC,Weight,pH,Age,Gender,Height,StaticWeight
0,132539,0.666667,0.222222,-0.695652,-0.218305,-0.6875,-0.333333,-0.374775,-0.333333,0.279561,...,-0.318182,-0.185185,1.818182,-0.471044,-0.302991,0.210159,-0.583612,0.0,-0.451583,-2.437098
1,132540,-0.518519,-0.055556,0.217391,-0.446695,0.1250,0.666667,-0.361386,0.666667,-0.704886,...,0.409091,1.259259,1.272727,0.128322,-0.019384,-0.351185,0.669324,1.0,0.353422,0.139491
2,132541,1.074074,2.555556,5.217391,-1.588648,-1.0000,7.000000,1.687192,-1.000000,0.853822,...,0.318182,-0.296296,-0.409091,-0.962831,-1.089358,1.519962,-1.153129,0.0,-0.451583,-0.637229
3,132543,1.074074,-0.944444,-1.260870,3.207554,-0.5625,-1.666667,-0.160545,-0.333333,0.033449,...,-0.113636,-0.074074,4.272727,-0.701569,0.109529,-0.164070,0.213711,1.0,0.670353,0.233071
4,132545,0.037037,-0.166667,-0.130435,0.695258,0.3750,-0.666667,0.107243,0.166667,-0.294700,...,-0.250000,-0.074074,0.000000,-1.177988,-0.822938,0.023044,1.352744,0.0,-0.939658,-2.437098


In [3]:
# Logistic Regression baseline - add balanced because of low prevalence
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report)

# Prepare arrays (drop RecordID as it has no predictive value)
X_tr = X_train[ALL_FEATURES]
X_v  = X_val[ALL_FEATURES]
X_te = X_test[ALL_FEATURES]
y_tr = y_train["In-hospital_death"]
y_v  = y_val["In-hospital_death"]
y_te = y_test["In-hospital_death"]

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_tr, y_tr)

lr_proba_test = lr.predict_proba(X_te)[:, 1] #directly compute probabilites as we want to compute AuROC and AuPRC - extract the second probability, the one for dying

print(f"Test AUROC: {roc_auc_score(y_te, lr_proba_test):.4f}  |  AUPRC: {average_precision_score(y_te, lr_proba_test):.4f}")

Test AUROC: 0.8417  |  AUPRC: 0.4914


In [4]:
# Gradient Boosted Tree baseline with random initialization
from sklearn.ensemble import GradientBoostingClassifier

gbt = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42,
)
gbt.fit(X_tr, y_tr)



gbt_proba_test = gbt.predict_proba(X_te)[:, 1]


print(f"Test AUROC: {roc_auc_score(y_te, gbt_proba_test):.4f}")


Test AUROC: 0.8443


### 2.1.2 Hyperparameter Tuning with GridSearchCV
We use 5-fold stratified cross-validation on the training set, optimizing for AUROC - optimizing for average_precision lead to worse results in both categories. We didn't include this in the GridSearch anymore to reduce necessary compute. Now we can make use of the validation set for the hyperparameter tuning with gridsearch.

In [5]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression Grid Search
# Let choose if ElasticNet and regularization strength as well as if the model should reweight because of the imbalanced data
lr_param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "l1_ratio": [0, 0.25, 0.5, 0.75, 1.0],
    "class_weight": ["balanced", None],
}

lr_gs = GridSearchCV(
    LogisticRegression(solver="saga", max_iter=5000, random_state=42),
    param_grid=lr_param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)
lr_gs.fit(X_tr, y_tr)

print(lr_gs.best_params_)
print(f"Best CV AUROC: {lr_gs.best_score_:.4f}")

Fitting 5 folds for each of 60 candidates, totalling 300 fits
{'C': 0.01, 'class_weight': 'balanced', 'l1_ratio': 0.25}
Best CV AUROC: 0.8293


In [16]:
# GBT Grid Search
gbt_param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [5, 6],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.6, 0.7],
    "min_samples_leaf": [15, 20, 25],
}

gbt_gs = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid=gbt_param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
)
gbt_gs.fit(X_tr, y_tr)

print(gbt_gs.best_params_)
print(f"Best CV AUROC: {gbt_gs.best_score_:.4f}")

Fitting 5 folds for each of 48 candidates, totalling 240 fits
{'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 20, 'n_estimators': 100, 'subsample': 0.7}
Best CV AUROC: 0.8552


In [14]:
# Evaluate tuned models on val & test
from matplotlib import pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

lr_best  = lr_gs.best_estimator_
gbt_best = gbt_gs.best_estimator_

results = {}
print(f"{'Model':15s}  {'Val AUROC':>10s}  {'Val AUPRC':>10s}  {'Test AUROC':>11s}  {'Test AUPRC':>11s}")
for name, model in [("LR (tuned)", lr_best), ("GBT (tuned)", gbt_best)]:
    p_val  = model.predict_proba(X_v)[:, 1]
    p_test = model.predict_proba(X_te)[:, 1]
    auc_val  = roc_auc_score(y_v, p_val)
    auc_test = roc_auc_score(y_te, p_test)
    prc_val  = average_precision_score(y_v, p_val)
    prc_test = average_precision_score(y_te, p_test)
    results[name] = {"val_auc": auc_val, "test_auc": auc_test,
                     "val_prc": prc_val, "test_prc": prc_test, "proba_test": p_test}
    print(f"{name:15s}  {auc_val:10.4f}  {prc_val:10.4f}  {auc_test:11.4f}  {prc_test:11.4f}")


Model             Val AUROC   Val AUPRC   Test AUROC   Test AUPRC
LR (tuned)           0.8506      0.5276       0.8460       0.4986
GBT (tuned)          0.8502      0.5204       0.8495       0.5270


### 2.1.3 Feature Engineering

The simple last-value features only use the final hour of each variable, discarding 47 hours of temporal information. We engineer additional features to capture:

- **Last value**: the most recent measurement (already used above)
- **First value**: admission baseline
- **First–last delta**: trajectory over the full 48h stay
- **Min / Max**: worst values during the stay
- **Mean / Std**: average level and variability (instability signals)
- **Missingness count**: number of hours with an actual measurement (sicker patients get tested more often)

#### Selected Features from Signal Processing

- **Absolute sum of changes(volatility)**
- **Linear trend slope**
- **Kurtosis (detects spiky outlier events)**
- **Count above/below mean (time spent in abnormal territory)**

In [17]:
# Feature Engineering
# Use scaled+imputed data for all features exept std, which is computed on raw data for caputring true variability.

# Data before imputation & scaling) for std calculation
df_train_raw = pd.read_parquet("processed/set_a.parquet")
df_val_raw   = pd.read_parquet("processed/set_b.parquet")
df_test_raw  = pd.read_parquet("processed/set_c.parquet")

def extract_engineered_features(df_scaled, df_raw):

    grp = df_scaled.groupby("RecordID")
    grp_raw = df_raw.groupby("RecordID")
    
    last  = grp[DYNAMIC_VARS].last()
    first = grp[DYNAMIC_VARS].first()
    mean  = grp[DYNAMIC_VARS].mean()
    vmin  = grp[DYNAMIC_VARS].min()
    vmax  = grp[DYNAMIC_VARS].max()
    delta = last - first
    std   = grp_raw[DYNAMIC_VARS].std().fillna(0)
    
    # Some signal-processing features suggested in the handout
    # 1. Absolute sum of changes (volatility)
    abs_sum_chg = grp[DYNAMIC_VARS].apply(lambda x: x.diff().abs().sum())
    
    # 2. Linear trend slope
    def linear_slope(series):
        y = series.dropna().values
        if len(y) < 2:
            return 0.0
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]
    
    trend = grp[DYNAMIC_VARS].agg(linear_slope)
    
    # 3. Kurtosis (detects spiky outlier events)
    kurt = grp[DYNAMIC_VARS].apply(lambda x: x.kurtosis()).fillna(0)
    
    # 4. Count above/below mean (time spent in abnormal territory)
    count_above = grp[DYNAMIC_VARS].apply(lambda x: (x > x.mean()).sum())
    count_below = grp[DYNAMIC_VARS].apply(lambda x: (x < x.mean()).sum())
    
    # New column names
    last        = last.rename(columns={c: f"{c}_last" for c in DYNAMIC_VARS})
    first       = first.rename(columns={c: f"{c}_first" for c in DYNAMIC_VARS})
    mean        = mean.rename(columns={c: f"{c}_mean" for c in DYNAMIC_VARS})
    std         = std.rename(columns={c: f"{c}_std" for c in DYNAMIC_VARS})
    vmin        = vmin.rename(columns={c: f"{c}_min" for c in DYNAMIC_VARS})
    vmax        = vmax.rename(columns={c: f"{c}_max" for c in DYNAMIC_VARS})
    delta       = delta.rename(columns={c: f"{c}_delta" for c in DYNAMIC_VARS})
    abs_sum_chg = abs_sum_chg.rename(columns={c: f"{c}_abs_sum_chg" for c in DYNAMIC_VARS})
    trend       = trend.rename(columns={c: f"{c}_trend" for c in DYNAMIC_VARS})
    kurt        = kurt.rename(columns={c: f"{c}_kurtosis" for c in DYNAMIC_VARS})
    count_above = count_above.rename(columns={c: f"{c}_count_above" for c in DYNAMIC_VARS})
    count_below = count_below.rename(columns={c: f"{c}_count_below" for c in DYNAMIC_VARS})
    
    statics = grp[STATIC_VARS].first()
    labels  = grp["In-hospital_death"].first().reset_index()
    
    features = pd.concat([last, first, mean, std, vmin, vmax, delta,
                           abs_sum_chg, trend, kurt, count_above, count_below,
                           statics], axis=1)
    features = features.reset_index()
    
    return features, labels

X_train_eng, y_train_eng = extract_engineered_features(df_train, df_train_raw)
X_val_eng,   y_val_eng   = extract_engineered_features(df_val,   df_val_raw)
X_test_eng,  y_test_eng  = extract_engineered_features(df_test,  df_test_raw)

ENG_FEATURES = [c for c in X_train_eng.columns if c != "RecordID"]

n_new = 5 * len(DYNAMIC_VARS)
print(f"Engineered feature count: {len(ENG_FEATURES)}")
print(f"  = 37 dynamic × 12 aggregations + {len(STATIC_VARS)} static = {37*12 + len(STATIC_VARS)}")
print(f"  New features (+{n_new}): abs_sum_chg, trend, kurtosis, count_above, count_below")
print(f"Train: {X_train_eng.shape}, Val: {X_val_eng.shape}, Test: {X_test_eng.shape}")
print(f"\nStd source: raw (pre-imputation) data")
print(f"All other features: scaled/imputed data")
print(f"\nNaN count: {X_train_eng[ENG_FEATURES].isna().sum().sum()}")
X_train_eng.head()

Engineered feature count: 448
  = 37 dynamic × 12 aggregations + 4 static = 448
  New features (+185): abs_sum_chg, trend, kurtosis, count_above, count_below
Train: (4000, 449), Val: (4000, 449), Test: (4000, 449)

Std source: raw (pre-imputation) data
All other features: scaled/imputed data

NaN count: 0


,RecordID,ALP_last,ALT_last,AST_last,Albumin_last,BUN_last,Bilirubin_last,Cholesterol_last,Creatinine_last,DiasABP_last,...,TroponinI_count_below,TroponinT_count_below,Urine_count_below,WBC_count_below,Weight_count_below,pH_count_below,Age,Gender,Height,StaticWeight
0,132539,0.666667,0.222222,-0.695652,-0.218305,-0.6875,-0.333333,-0.374775,-0.333333,0.279561,...,0,49,33,15,0,0,-0.583612,0.0,-0.451583,-2.437098
1,132540,-0.518519,-0.055556,0.217391,-0.446695,0.1250,0.666667,-0.361386,0.666667,-0.704886,...,49,0,32,13,20,12,0.669324,1.0,0.353422,0.139491
2,132541,1.074074,2.555556,5.217391,-1.588648,-1.0000,7.000000,1.687192,-1.000000,0.853822,...,0,0,31,38,0,16,-1.153129,0.0,-0.451583,-0.637229
3,132543,1.074074,-0.944444,-1.260870,3.207554,-0.5625,-1.666667,-0.160545,-0.333333,0.033449,...,49,49,20,40,0,49,0.213711,1.0,0.670353,0.233071
4,132545,0.037037,-0.166667,-0.130435,0.695258,0.3750,-0.666667,0.107243,0.166667,-0.294700,...,0,49,37,43,49,0,1.352744,0.0,-0.939658,-2.437098


In [18]:
# Scaling engineered features
from sklearn.preprocessing import StandardScaler

scaler_eng = StandardScaler()
X_tr_eng = scaler_eng.fit_transform(X_train_eng[ENG_FEATURES])
X_v_eng  = scaler_eng.transform(X_val_eng[ENG_FEATURES])
X_te_eng = scaler_eng.transform(X_test_eng[ENG_FEATURES])

y_tr_eng = y_train_eng["In-hospital_death"].values
y_v_eng  = y_val_eng["In-hospital_death"].values
y_te_eng = y_test_eng["In-hospital_death"].values

print(f"Scaled feature matrices — Train: {X_tr_eng.shape}, Val: {X_v_eng.shape}, Test: {X_te_eng.shape}")

Scaled feature matrices — Train: (4000, 448), Val: (4000, 448), Test: (4000, 448)


In [ ]:
#Train LR & GBT on engineered features with GridSearchCV

# LR
lr_gs_eng = GridSearchCV(
    LogisticRegression(solver="saga", max_iter=7000, random_state=42),
    param_grid={
        "C": [0.001, 0.01, 0.1, 1, 10],
        "l1_ratio": [0, 0.25, 0.5, 0.75, 1.0],
        "class_weight": ["balanced", None],
    },
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1,
)
lr_gs_eng.fit(X_tr_eng, y_tr_eng)
print(f"\nLR best params: {lr_gs_eng.best_params_}")
print(f"LR best CV AUROC: {lr_gs_eng.best_score_:.4f}")

# GBT
gbt_gs_eng = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid={
        "n_estimators": [100, 200, 500],
        "max_depth": [ 5, 6],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.7, 0.8],
        "min_samples_leaf": [10, 15],
    },
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1,
)
gbt_gs_eng.fit(X_tr_eng, y_tr_eng)
print(f"\nGBT best params: {gbt_gs_eng.best_params_}")
print(f"GBT best CV AUROC: {gbt_gs_eng.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/Users/ferdinandunterhuber/ETH/Machine-Learning-for-Health-Care/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/ferdinandunterhuber/ETH/Machine-Learning-for-Health-Care/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/ferdinandunterhuber/ETH/Machine-Learning-for-Health-Care/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/ferdinandunterhuber/ETH/Machine-Learning-for-Health-Care/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/ferdinandunterhuber/ETH/Machine-Learning-for-Health-Care/


LR best params: {'C': 0.01, 'class_weight': 'balanced', 'l1_ratio': 0.25}
LR best CV AUROC: 0.8391
Fitting 5 folds for each of 72 candidates, totalling 360 fits


### Feature Selection via ElasticNet

The tuned LR with L1/ElasticNet regularization shrinks unimportant coefficients of tbe previous 448 variables to zero. We extract the non-zero features and retrain both LR and GBT on only those — reducing noise, multicollinearity (for LR) and potentially improving generalization.

In [ ]:
# ── Extract non-zero features from ElasticNet LR ─────────────────────────────
lr_coefs = lr_gs_eng.coef_[0]
nonzero_mask = lr_coefs != 0
selected_features = [f for f, m in zip(ENG_FEATURES, nonzero_mask) if m]

print(f"ElasticNet LR selected {len(selected_features)} / {len(ENG_FEATURES)} features")
print(f"Dropped {len(ENG_FEATURES) - len(selected_features)} features (coef = 0)\n")

# Show top features by absolute coefficient
coef_series = pd.Series(lr_coefs, index=ENG_FEATURES)
top_features = coef_series[nonzero_mask].abs().sort_values(ascending=False).head(20)
print("Top 20 features by |coefficient|:")
for feat, val in top_features.items():
    sign = "+" if coef_series[feat] > 0 else "-"
    print(f"  {sign} {feat:<30s}  |coef| = {val:.4f}")

In [ ]:
# ── Retrain LR & GBT on selected features only ──────────────────────────────
sel_idx = [ENG_FEATURES.index(f) for f in selected_features]

X_tr_sel = X_tr_eng[:, sel_idx]
X_v_sel  = X_v_eng[:, sel_idx]
X_te_sel = X_te_eng[:, sel_idx]

# LR on selected features
lr_gs_sel = GridSearchCV(
    LogisticRegression(solver="saga", max_iter=5000, random_state=42),
    param_grid={
        "C": [0.001, 0.01, 0.1],
        "l1_ratio": [0, 0.25, 0.5, 0.75, 1.0],
        "class_weight": ["balanced", None],
    },
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1,
)
lr_gs_sel.fit(X_tr_sel, y_tr_eng)

# GBT on selected features
gbt_gs_sel = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid={
        "n_estimators": [100, 200, 500],
        "max_depth": [ 4, 5],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.7, 0.8],
        "min_samples_leaf": [10, 20],
    },
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1,
)
gbt_gs_sel.fit(X_tr_sel, y_tr_eng)

print(f"\nLR  (selected) best CV AUROC: {lr_gs_sel.best_score_:.4f}  params: {lr_gs_sel.best_params_}")
print(f"GBT (selected) best CV AUROC: {gbt_gs_sel.best_score_:.4f}  params: {gbt_gs_sel.best_params_}")

## 2.2 RNN

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

# ── 1. Reshape parquet → 3-D tensors ────────────────────────────────────────
# Dynamic: (N, 49, 37)  — 49 time steps, 37 features
# Static:  (N, 4)       — Age, Gender, Height, StaticWeight (constant per patient)

def df_to_tensors(df, dynamic_vars, static_vars, n_steps=49):
    df = df.sort_values(["RecordID", "hour"])
    n_patients = df["RecordID"].nunique()
    X_dyn  = torch.tensor(df[dynamic_vars].values.reshape(n_patients, n_steps, -1), dtype=torch.float32)
    X_stat = torch.tensor(df.groupby("RecordID")[static_vars].first().values, dtype=torch.float32)
    y      = torch.tensor(df.groupby("RecordID")["In-hospital_death"].first().values, dtype=torch.float32)
    return X_dyn, X_stat, y

X_dyn_tr, X_stat_tr, y_tr_rnn = df_to_tensors(df_train, DYNAMIC_VARS, STATIC_VARS)
X_dyn_v,  X_stat_v,  y_v_rnn  = df_to_tensors(df_val,   DYNAMIC_VARS, STATIC_VARS)
X_dyn_te, X_stat_te, y_te_rnn = df_to_tensors(df_test,  DYNAMIC_VARS, STATIC_VARS)

print(f"Dynamic  — Train: {X_dyn_tr.shape}  Val: {X_dyn_v.shape}  Test: {X_dyn_te.shape}")
print(f"Static   — Train: {X_stat_tr.shape}   Val: {X_stat_v.shape}   Test: {X_stat_te.shape}")
print(f"Positive rate: {y_tr_rnn.mean():.3f}")

# ── 2. DataLoaders ──────────────────────────────────────────────────────────
BATCH = 128
train_dl = DataLoader(TensorDataset(X_dyn_tr, X_stat_tr, y_tr_rnn), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_dyn_v,  X_stat_v,  y_v_rnn),  batch_size=BATCH)
test_dl  = DataLoader(TensorDataset(X_dyn_te, X_stat_te, y_te_rnn), batch_size=BATCH)

# ── 3. LSTM model (static features fused after LSTM) ────────────────────────
class LSTM(nn.Module):
    def __init__(self, input_dim=37, static_dim=4, hidden_dim=128, n_layers=1, dropout=0.3):
        super().__init__()
        self.static_dim = static_dim
        # Static features are concatenated to every time step
        self.lstm = nn.LSTM(
            input_size=input_dim + static_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x_dyn, x_stat):
        # x_dyn: (batch, 49, 37),  x_stat: (batch, 4)
        # Broadcast static features to every time step
        x_stat_exp = x_stat.unsqueeze(1).expand(-1, x_dyn.size(1), -1)  # (batch, 49, 4)
        x = torch.cat([x_dyn, x_stat_exp], dim=2)  # (batch, 49, 41)
        _, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]                 # (batch, hidden)
        return self.head(last_hidden).squeeze(-1)

device = torch.device("mps" if torch.backends.mps.is_available() else
                       "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = LSTM().to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ── 4. Training ─────────────────────────────────────────────────────────────
pos_weight = torch.tensor([(y_tr_rnn == 0).sum() / (y_tr_rnn == 1).sum()]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

EPOCHS = 50
best_val_auc = 0
patience_ctr, patience_limit = 0, 10
history = {"train_loss": [], "val_auc": [], "val_prc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for xd, xs, yb in train_dl:
        xd, xs, yb = xd.to(device), xs.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xd, xs), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(xd)
    avg_loss = total_loss / len(train_dl.dataset)

    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in val_dl:
            preds.append(torch.sigmoid(model(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    val_auc = roc_auc_score(labels, preds)
    val_prc = average_precision_score(labels, preds)
    scheduler.step(val_auc)

    history["train_loss"].append(avg_loss)
    history["val_auc"].append(val_auc)
    history["val_prc"].append(val_prc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or patience_ctr == 0:
        print(f"Epoch {epoch:3d}  loss={avg_loss:.4f}  val_AUROC={val_auc:.4f}  val_AUPRC={val_prc:.4f}  lr={optimizer.param_groups[0]['lr']:.1e}")

    if patience_ctr >= patience_limit:
        print(f"Early stopping at epoch {epoch}")
        break

# ── 5. Test evaluation ──────────────────────────────────────────────────────
model.load_state_dict(best_state)
model.eval()
preds_test, labels_test = [], []
with torch.no_grad():
    for xd, xs, yb in test_dl:
        preds_test.append(torch.sigmoid(model(xd.to(device), xs.to(device))).cpu())
        labels_test.append(yb)
preds_test  = torch.cat(preds_test).numpy()
labels_test = torch.cat(labels_test).numpy()

test_auc = roc_auc_score(labels_test, preds_test)
test_prc = average_precision_score(labels_test, preds_test)
print(f"\n{'LSTM (unidirectional)':<30s}  Val AUROC={best_val_auc:.4f}  Test AUROC={test_auc:.4f}  Test AUPRC={test_prc:.4f}")


In [ ]:
# ── Bidirectional LSTM ────────────────────────────────────────────────────────

class BiLSTM(nn.Module):
    def __init__(self, input_dim=37, static_dim=4, hidden_dim=128, n_layers=1, dropout=0.5):
        super().__init__()
        self.static_dim = static_dim
        self.lstm = nn.LSTM(
            input_size=input_dim + static_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.head = nn.Sequential(
            nn.Linear(2 * hidden_dim, 1),
        )

    def forward(self, x_dyn, x_stat):
        # x_dyn: (batch, 49, 37),  x_stat: (batch, 4)
        x_stat_exp = x_stat.unsqueeze(1).expand(-1, x_dyn.size(1), -1)  # (batch, 49, 4)
        x = torch.cat([x_dyn, x_stat_exp], dim=2)  # (batch, 49, 41)
        _, (h_n, _) = self.lstm(x)
        fwd = h_n[-2]   # last layer, forward
        bwd = h_n[-1]   # last layer, backward
        combined = torch.cat([fwd, bwd], dim=1)  # (batch, 2*hidden)
        return self.head(combined).squeeze(-1)

bi_model = BiLSTM().to(device)
print(bi_model)
print(f"Parameters: {sum(p.numel() for p in bi_model.parameters()):,}")

# ── Training ────────────────────────────────────────────────────────────────
bi_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
bi_optimizer = torch.optim.Adam(bi_model.parameters(), lr=5e-4, weight_decay=1e-3)
bi_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(bi_optimizer, patience=5, factor=0.5)

best_bi_auc = 0
bi_patience, bi_patience_limit = 0, 10
bi_history = {"train_loss": [], "val_auc": [], "val_prc": []}

for epoch in range(1, EPOCHS + 1):
    bi_model.train()
    total_loss = 0
    for xd, xs, yb in train_dl:
        xd, xs, yb = xd.to(device), xs.to(device), yb.to(device)
        bi_optimizer.zero_grad()
        loss = bi_criterion(bi_model(xd, xs), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bi_model.parameters(), 1.0)
        bi_optimizer.step()
        total_loss += loss.item() * len(xd)
    avg_loss = total_loss / len(train_dl.dataset)

    bi_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in val_dl:
            preds.append(torch.sigmoid(bi_model(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    val_auc = roc_auc_score(labels, preds)
    val_prc = average_precision_score(labels, preds)
    bi_scheduler.step(val_auc)

    bi_history["train_loss"].append(avg_loss)
    bi_history["val_auc"].append(val_auc)
    bi_history["val_prc"].append(val_prc)

    if val_auc > best_bi_auc:
        best_bi_auc = val_auc
        best_bi_state = {k: v.cpu().clone() for k, v in bi_model.state_dict().items()}
        bi_patience = 0
    else:
        bi_patience += 1

    if epoch % 5 == 0 or bi_patience == 0:
        print(f"Epoch {epoch:3d}  loss={avg_loss:.4f}  val_AUROC={val_auc:.4f}  val_AUPRC={val_prc:.4f}  lr={bi_optimizer.param_groups[0]['lr']:.1e}")

    if bi_patience >= bi_patience_limit:
        print(f"Early stopping at epoch {epoch}")
        break

# ── Test evaluation ─────────────────────────────────────────────────────────
bi_model.load_state_dict(best_bi_state)
bi_model.eval()
bi_preds_test, bi_labels_test = [], []
with torch.no_grad():
    for xd, xs, yb in test_dl:
        bi_preds_test.append(torch.sigmoid(bi_model(xd.to(device), xs.to(device))).cpu())
        bi_labels_test.append(yb)
bi_preds_test  = torch.cat(bi_preds_test).numpy()
bi_labels_test = torch.cat(bi_labels_test).numpy()

bi_test_auc = roc_auc_score(bi_labels_test, bi_preds_test)
bi_test_prc = average_precision_score(bi_labels_test, bi_preds_test)


## 2.3 a Transformer

In [ ]:
# ── Transformer for Mortality Prediction ──────────────────────────────────────
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(position * div_term)
        else:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class Transformer(nn.Module):
    def __init__(self, input_dim=37, static_dim=4, d_model=64, nhead=4,
                 n_layers=2, dim_ff=128, dropout=0.3):
        super().__init__()
        # Project input features to d_model
        self.input_proj = nn.Linear(input_dim + static_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=50)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Learnable [CLS] token for classification
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.head = nn.Linear(d_model, 1)

    def forward(self, x_dyn, x_stat):
        # x_dyn: (batch, 49, 37), x_stat: (batch, 4)
        B, T, _ = x_dyn.shape

        # Concat static features at every time step
        x_stat_exp = x_stat.unsqueeze(1).expand(-1, T, -1)  # (B, 49, 4)
        x = torch.cat([x_dyn, x_stat_exp], dim=2)           # (B, 49, 41)

        # Project to d_model and add positional encoding
        x = self.dropout(self.pos_enc(self.input_proj(x)))   # (B, 49, d_model)

        # Prepend [CLS] token
        cls = self.cls_token.expand(B, -1, -1)               # (B, 1, d_model)
        x = torch.cat([cls, x], dim=1)                       # (B, 50, d_model)

        # Transformer encoder
        x = self.transformer(x)                              # (B, 50, d_model)

        # Classify from [CLS] token
        return self.head(x[:, 0]).squeeze(-1)

tx_model = Transformer().to(device)
print(tx_model)
print(f"Parameters: {sum(p.numel() for p in tx_model.parameters()):,}")

# ── Training ────────────────────────────────────────────────────────────────
tx_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
tx_optimizer = torch.optim.Adam(tx_model.parameters(), lr=5e-4, weight_decay=1e-4)
tx_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(tx_optimizer, patience=5, factor=0.5)

best_tx_auc = 0
tx_patience, tx_patience_limit = 0, 10
tx_history = {"train_loss": [], "val_auc": [], "val_prc": []}

for epoch in range(1, EPOCHS + 1):
    tx_model.train()
    total_loss = 0
    for xd, xs, yb in train_dl:
        xd, xs, yb = xd.to(device), xs.to(device), yb.to(device)
        tx_optimizer.zero_grad()
        loss = tx_criterion(tx_model(xd, xs), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tx_model.parameters(), 1.0)
        tx_optimizer.step()
        total_loss += loss.item() * len(xd)
    avg_loss = total_loss / len(train_dl.dataset)

    tx_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in val_dl:
            preds.append(torch.sigmoid(tx_model(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    val_auc = roc_auc_score(labels, preds)
    val_prc = average_precision_score(labels, preds)
    tx_scheduler.step(val_auc)

    tx_history["train_loss"].append(avg_loss)
    tx_history["val_auc"].append(val_auc)
    tx_history["val_prc"].append(val_prc)

    if val_auc > best_tx_auc:
        best_tx_auc = val_auc
        best_tx_state = {k: v.cpu().clone() for k, v in tx_model.state_dict().items()}
        tx_patience = 0
    else:
        tx_patience += 1

    if epoch % 5 == 0 or tx_patience == 0:
        print(f"Epoch {epoch:3d}  loss={avg_loss:.4f}  val_AUROC={val_auc:.4f}  val_AUPRC={val_prc:.4f}  lr={tx_optimizer.param_groups[0]['lr']:.1e}")

    if tx_patience >= tx_patience_limit:
        print(f"Early stopping at epoch {epoch}")
        break

# ── Test evaluation ─────────────────────────────────────────────────────────
tx_model.load_state_dict(best_tx_state)
tx_model.eval()
tx_preds_test, tx_labels_test = [], []
with torch.no_grad():
    for xd, xs, yb in test_dl:
        tx_preds_test.append(torch.sigmoid(tx_model(xd.to(device), xs.to(device))).cpu())
        tx_labels_test.append(yb)
tx_preds_test  = torch.cat(tx_preds_test).numpy()
tx_labels_test = torch.cat(tx_labels_test).numpy()

tx_test_auc = roc_auc_score(tx_labels_test, tx_preds_test)
tx_test_prc = average_precision_score(tx_labels_test, tx_preds_test)


print(f"\nTransformer  Val AUROC={best_tx_auc:.4f}  Test AUROC={tx_test_auc:.4f}  Test AUPRC={tx_test_prc:.4f}")

In [ ]:
# ── Hyperparameter Tuning for all DL models ──────────────────────────────────
from itertools import product

def train_and_eval(model, train_dl, val_dl, device, lr, wd, epochs=50, patience_limit=10):
    """Train a model, return best val AUROC, AUPRC, and state dict."""
    pos_w = torch.tensor([(y_tr_rnn == 0).sum() / (y_tr_rnn == 1).sum()]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_auc, best_state, patience_ctr = 0, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for xd, xs, yb in train_dl:
            xd, xs, yb = xd.to(device), xs.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xd, xs), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for xd, xs, yb in val_dl:
                preds.append(torch.sigmoid(model(xd.to(device), xs.to(device))).cpu())
                labels.append(yb)
        preds  = torch.cat(preds).numpy()
        labels = torch.cat(labels).numpy()
        val_auc = roc_auc_score(labels, preds)
        scheduler.step(val_auc)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if patience_ctr >= patience_limit:
            break

    # Eval best model on val for AUPRC too
    model.load_state_dict(best_state)
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in val_dl:
            preds.append(torch.sigmoid(model(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    best_prc = average_precision_score(labels, preds)
    return best_auc, best_prc, best_state

# ── Hyperparameter grids ────────────────────────────────────────────────────
lstm_grid = {
    "hidden_dim": [64, 128],
    "n_layers":   [1],
    "dropout":    [0.3, 0.5],
    "lr":         [1e-3, 5e-4],
    "wd":         [1e-3],
}

transformer_grid = {
    "d_model":  [64],
    "nhead":    [4],
    "n_layers": [2],
    "dim_ff":   [128, 256],
    "dropout":  [0.2, 0.3],
    "lr":       [5e-4, 1e-3],
    "wd":       [1e-4, 1e-3],
}

# Tune LSTM
print("=" * 70)
print("Tuning LSTM ")
print("=" * 70)
best_lstm_result = {"auc": 0}
lstm_keys = list(lstm_grid.keys())
for vals in product(*lstm_grid.values()):
    cfg = dict(zip(lstm_keys, vals))
    m = LSTM(
        hidden_dim=cfg["hidden_dim"], n_layers=cfg["n_layers"], dropout=cfg["dropout"]
    ).to(device)
    auc, prc, state = train_and_eval(m, train_dl, val_dl, device, cfg["lr"], cfg["wd"])
    print(f"  h={cfg['hidden_dim']:3d} layers={cfg['n_layers']} drop={cfg['dropout']:.1f} "
          f"lr={cfg['lr']:.0e} wd={cfg['wd']:.0e}  → AUROC={auc:.4f} AUPRC={prc:.4f}")
    if auc > best_lstm_result["auc"]:
        best_lstm_result = {"auc": auc, "prc": prc, "state": state, "cfg": cfg}

print(f"\n  Best LSTM: AUROC={best_lstm_result['auc']:.4f} AUPRC={best_lstm_result['prc']:.4f}")
print(f"  Config: {best_lstm_result['cfg']}")

# ── Tune BiLSTM ─────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("Tuning BiLSTM")
print("=" * 70)
best_bilstm_result = {"auc": 0}
for vals in product(*lstm_grid.values()):
    cfg = dict(zip(lstm_keys, vals))
    m = BiLSTM(
        hidden_dim=cfg["hidden_dim"], n_layers=cfg["n_layers"], dropout=cfg["dropout"]
    ).to(device)
    auc, prc, state = train_and_eval(m, train_dl, val_dl, device, cfg["lr"], cfg["wd"])
    print(f"  h={cfg['hidden_dim']:3d} layers={cfg['n_layers']} drop={cfg['dropout']:.1f} "
          f"lr={cfg['lr']:.0e} wd={cfg['wd']:.0e}  → AUROC={auc:.4f} AUPRC={prc:.4f}")
    if auc > best_bilstm_result["auc"]:
        best_bilstm_result = {"auc": auc, "prc": prc, "state": state, "cfg": cfg}

print(f"\n  Best BiLSTM: AUROC={best_bilstm_result['auc']:.4f} AUPRC={best_bilstm_result['prc']:.4f}")
print(f"  Config: {best_bilstm_result['cfg']}")

# ── Tune Transformer ────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("Tuning Transformer")
print("=" * 70)
best_tx_result = {"auc": 0}
tx_keys = list(transformer_grid.keys())
for vals in product(*transformer_grid.values()):
    cfg = dict(zip(tx_keys, vals))
    # nhead must divide d_model
    if cfg["d_model"] % cfg["nhead"] != 0:
        continue
    m = Transformer(
        d_model=cfg["d_model"], nhead=cfg["nhead"], n_layers=cfg["n_layers"],
        dim_ff=cfg["dim_ff"], dropout=cfg["dropout"]
    ).to(device)
    auc, prc, state = train_and_eval(m, train_dl, val_dl, device, cfg["lr"], cfg["wd"])
    print(f"  d={cfg['d_model']:3d} heads={cfg['nhead']} layers={cfg['n_layers']} ff={cfg['dim_ff']:3d} "
          f"drop={cfg['dropout']:.1f} lr={cfg['lr']:.0e} wd={cfg['wd']:.0e}  → AUROC={auc:.4f} AUPRC={prc:.4f}")
    if auc > best_tx_result["auc"]:
        best_tx_result = {"auc": auc, "prc": prc, "state": state, "cfg": cfg}

print(f"\n  Best Transformer: AUROC={best_tx_result['auc']:.4f} AUPRC={best_tx_result['prc']:.4f}")
print(f"  Config: {best_tx_result['cfg']}")

# ── Final test evaluation with best configs ─────────────────────────────────
print("\n" + "=" * 70)
print("Test set evaluation with best hyperparameters")
print("=" * 70)

results_tuned = []
for name, ModelClass, result, is_tx in [
    ("LSTM (tuned)",   LSTM,       best_lstm_result,   False),
    ("BiLSTM (tuned)", BiLSTM,     best_bilstm_result, False),
    ("Transformer (tuned)", Transformer, best_tx_result, True),
]:
    cfg = result["cfg"]
    if is_tx:
        m = ModelClass(d_model=cfg["d_model"], nhead=cfg["nhead"],
                       n_layers=cfg["n_layers"], dim_ff=cfg["dim_ff"],
                       dropout=cfg["dropout"]).to(device)
    else:
        m = ModelClass(hidden_dim=cfg["hidden_dim"], n_layers=cfg["n_layers"],
                       dropout=cfg["dropout"]).to(device)
    m.load_state_dict(result["state"])
    m.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in test_dl:
            preds.append(torch.sigmoid(m(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    pt = torch.cat(preds).numpy()
    yt = torch.cat(labels).numpy()
    t_auc = roc_auc_score(yt, pt)
    t_prc = average_precision_score(yt, pt)
    results_tuned.append((name, result["auc"], t_auc, t_prc, pt, yt))

print(f"\n{'Model':<30s} {'Val AUROC':>10s} {'Test AUROC':>11s} {'Test AUPRC':>11s}")
print("-" * 65)
for name, v_auc, t_auc, t_prc, _, _ in results_tuned:
    print(f"{name:<30s} {v_auc:10.4f} {t_auc:11.4f} {t_prc:11.4f}")


## 2.3 b Set-based Representation (Horn et al.)

Instead of modeling a sequence of time steps (with imputed missing values), we model a
**sequence of actual measurements**. Each observation is encoded as a triplet **(t, z, v)**:
- **t** — scaled time in [0, 1]
- **z** — one-hot encoding of the variable (41 categories: 37 dynamic + 4 static)
- **v** — the scaled measurement value

This avoids imputation entirely and naturally handles irregular sampling. We feed the
triplet sequence into a Transformer.

In [ ]:
# ── Triplet (t, z, v) encoding — Horn et al. ─────────────────────────────────
import pickle
from torch.nn.utils.rnn import pad_sequence

# Load the scaler fitted in Q1.3
with open("processed/scalers.pkl", "rb") as f:
    scalers_dict = pickle.load(f)

ALL_VARS = DYNAMIC_VARS + STATIC_VARS  # 41 variables
var_to_idx = {v: i for i, v in enumerate(ALL_VARS)}
N_VARS = len(ALL_VARS)
TRIPLET_DIM = 2 + N_VARS  # 1 (time) + 1 (value) + 41 (one-hot) = 43

def build_triplets(df_raw):
    """Raw data → list of triplet arrays per patient + labels."""
    t_max = df_raw["hour"].max()
    triplets, labels = [], []

    for rid, grp in df_raw.groupby("RecordID"):
        obs = []
        labels.append(grp["In-hospital_death"].iloc[0])

        # Dynamic: one triplet per actual measurement (skip NaN)
        for _, row in grp.iterrows():
            t = row["hour"] / t_max
            for var in DYNAMIC_VARS:
                if pd.notna(row[var]):
                    z = np.zeros(N_VARS, dtype=np.float32)
                    z[var_to_idx[var]] = 1.0
                    v = float(row[var])
                    if var in scalers_dict:
                        v = float(scalers_dict[var].transform([[v]])[0, 0])
                    obs.append(np.concatenate([[t, v], z]))

        # Static: add once at t=0
        row0 = grp.iloc[0]
        for var in STATIC_VARS:
            if pd.notna(row0[var]):
                z = np.zeros(N_VARS, dtype=np.float32)
                z[var_to_idx[var]] = 1.0
                v = float(row0[var])
                if var in scalers_dict:
                    v = float(scalers_dict[var].transform([[v]])[0, 0])
                obs.append(np.concatenate([[0.0, v], z]))

        triplets.append(np.array(obs, dtype=np.float32) if obs else np.zeros((1, TRIPLET_DIM), dtype=np.float32))

    return triplets, np.array(labels, dtype=np.float32)

df_train_raw = pd.read_parquet("processed/set_a.parquet")
df_val_raw   = pd.read_parquet("processed/set_b.parquet")
df_test_raw  = pd.read_parquet("processed/set_c.parquet")

print("Building triplets...")
trip_tr, y_trip_tr = build_triplets(df_train_raw)
trip_v,  y_trip_v  = build_triplets(df_val_raw)
trip_te, y_trip_te = build_triplets(df_test_raw)

lengths = [len(t) for t in trip_tr]
print(f"Patients: {len(trip_tr)}")
print(f"Triplets per patient — min: {min(lengths)}, median: {int(np.median(lengths))}, max: {max(lengths)}")
print(f"Triplet dim: {TRIPLET_DIM}  (1 time + 1 value + {N_VARS} one-hot)")

In [ ]:
# ── Transformer on triplet sequences ─────────────────────────────────────────

def collate_triplets(batch):
    seqs, labels = zip(*batch)
    lengths = [len(s) for s in seqs]
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    # Padding mask: True where padded
    max_len = seqs_padded.size(1)
    mask = torch.arange(max_len).unsqueeze(0) >= torch.tensor(lengths).unsqueeze(1)
    return seqs_padded, mask, torch.tensor(labels, dtype=torch.float32)

class TripletDataset(torch.utils.data.Dataset):
    def __init__(self, triplets, labels):
        self.triplets = [torch.tensor(t, dtype=torch.float32) for t in triplets]
        self.labels = labels
    def __len__(self):
        return len(self.triplets)
    def __getitem__(self, idx):
        return self.triplets[idx], self.labels[idx]

trip_train_dl = DataLoader(TripletDataset(trip_tr, y_trip_tr), batch_size=64, shuffle=True,  collate_fn=collate_triplets)
trip_val_dl   = DataLoader(TripletDataset(trip_v,  y_trip_v),  batch_size=64, shuffle=False, collate_fn=collate_triplets)
trip_test_dl  = DataLoader(TripletDataset(trip_te, y_trip_te), batch_size=64, shuffle=False, collate_fn=collate_triplets)

import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

class TripletTransformer(nn.Module):
    def __init__(self, input_dim=TRIPLET_DIM, d_model=64, nhead=4,
                 n_layers=2, dim_ff=128, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x_padded, pad_mask):
        # x_padded: (B, L, 43), pad_mask: (B, L) True=padded
        B = x_padded.size(0)
        x = self.dropout(self.input_proj(x_padded))       # (B, L, d_model)
        cls = self.cls_token.expand(B, -1, -1)             # (B, 1, d_model)
        x = torch.cat([cls, x], dim=1)                     # (B, L+1, d_model)
        # Extend mask for CLS token (never masked)
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=pad_mask.device)
        mask = torch.cat([cls_mask, pad_mask], dim=1)       # (B, L+1)
        x = self.transformer(x, src_key_padding_mask=mask)
        return self.head(x[:, 0]).squeeze(-1)

trip_tx = TripletTransformer().to(device)
print(trip_tx)
print(f"Parameters: {sum(p.numel() for p in trip_tx.parameters()):,}")

# ── Training ────────────────────────────────────────────────────────────────
trip_pos_w = torch.tensor([(y_trip_tr == 0).sum() / (y_trip_tr == 1).sum()], dtype=torch.float32).to(device)
trip_crit = nn.BCEWithLogitsLoss(pos_weight=trip_pos_w)
trip_opt = torch.optim.Adam(trip_tx.parameters(), lr=5e-4, weight_decay=1e-3)
trip_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(trip_opt, patience=5, factor=0.5)

best_trip_auc = 0
trip_pat = 0
trip_history = {"train_loss": [], "val_auc": [], "val_prc": []}

for epoch in range(1, EPOCHS + 1):
    trip_tx.train()
    total_loss = 0
    for x_pad, mask, yb in trip_train_dl:
        x_pad, mask, yb = x_pad.to(device), mask.to(device), yb.to(device)
        trip_opt.zero_grad()
        loss = trip_crit(trip_tx(x_pad, mask), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trip_tx.parameters(), 1.0)
        trip_opt.step()
        total_loss += loss.item() * len(yb)
    avg_loss = total_loss / len(trip_train_dl.dataset)

    trip_tx.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x_pad, mask, yb in trip_val_dl:
            preds.append(torch.sigmoid(trip_tx(x_pad.to(device), mask.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    val_auc = roc_auc_score(labels, preds)
    val_prc = average_precision_score(labels, preds)
    trip_sched.step(val_auc)

    trip_history["train_loss"].append(avg_loss)
    trip_history["val_auc"].append(val_auc)
    trip_history["val_prc"].append(val_prc)

    if val_auc > best_trip_auc:
        best_trip_auc = val_auc
        best_trip_state = {k: v.cpu().clone() for k, v in trip_tx.state_dict().items()}
        trip_pat = 0
    else:
        trip_pat += 1
    if epoch % 5 == 0 or trip_pat == 0:
        print(f"Epoch {epoch:3d}  loss={avg_loss:.4f}  val_AUROC={val_auc:.4f}  val_AUPRC={val_prc:.4f}  lr={trip_opt.param_groups[0]['lr']:.1e}")
    if trip_pat >= 10:
        print(f"Early stopping at epoch {epoch}")
        break

# ── Test ────────────────────────────────────────────────────────────────────
trip_tx.load_state_dict(best_trip_state)
trip_tx.eval()
trip_preds_test, trip_labels_test = [], []
with torch.no_grad():
    for x_pad, mask, yb in trip_test_dl:
        trip_preds_test.append(torch.sigmoid(trip_tx(x_pad.to(device), mask.to(device))).cpu())
        trip_labels_test.append(yb)
trip_preds_test  = torch.cat(trip_preds_test).numpy()
trip_labels_test = torch.cat(trip_labels_test).numpy()

trip_test_auc = roc_auc_score(trip_labels_test, trip_preds_test)
trip_test_prc = average_precision_score(trip_labels_test, trip_preds_test)
print(f"\nTriplet Transformer  Val AUROC={best_trip_auc:.4f}  Test AUROC={trip_test_auc:.4f}  Test AUPRC={trip_test_prc:.4f}")

In [ ]:
# ── Final Comparison: All Deep Learning Models ───────────────────────────────
all_dl = [
    ("LSTM",               preds_test,      labels_test,      best_val_auc,  test_auc,      test_prc),
    ("BiLSTM",             bi_preds_test,   bi_labels_test,   best_bi_auc,   bi_test_auc,   bi_test_prc),
    ("Transformer",        tx_preds_test,   tx_labels_test,   best_tx_auc,   tx_test_auc,   tx_test_prc),
    ("Triplet Transformer",trip_preds_test, trip_labels_test,  best_trip_auc, trip_test_auc,  trip_test_prc),
]

print(f"{'Model':<30s} {'Val AUROC':>10s} {'Test AUROC':>11s} {'Test AUPRC':>11s}")
print("-" * 65)
for name, _, _, v, ta, tp in all_dl:
    print(f"{name:<30s} {v:10.4f} {ta:11.4f} {tp:11.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
baseline = y_te_rnn.mean().item()
for name, pt, yt, _, _, _ in all_dl:
    fpr, tpr, _ = roc_curve(yt, pt)
    prec, rec, _ = precision_recall_curve(yt, pt)
    axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(yt, pt):.3f})")
    axes[1].plot(rec, prec, label=f"{name} ({average_precision_score(yt, pt):.3f})")
axes[0].plot([0,1],[0,1], "k--", alpha=0.3)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC"); axes[0].legend()
axes[1].axhline(baseline, color="k", linestyle="--", alpha=0.4, label=f"Baseline ({baseline:.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("PRC"); axes[1].legend()
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, hist in [("LSTM", history), ("BiLSTM", bi_history),
                    ("Transformer", tx_history), ("Triplet TX", trip_history)]:
    axes[0].plot(hist["train_loss"], label=name)
    axes[1].plot(hist["val_auc"], label=name)
axes[0].set_title("Train Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].set_title("Val AUROC"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()